In [ ]:
#%pip install ollama

Note: you may need to restart the kernel to use updated packages.


## 이미지 분석 예제

In [3]:
import ollama
import gradio as gr
import base64
from PIL import Image
import io

def encode_image(image_path): # 이미지 경로를 받으면 인코딩시켜줌
    """Encode image to base64 string"""
    with open(image_path, "rb") as image_file:     # rb: 바이너리 읽기(이미지는 0,1 형태의 바이너리)
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_image_with_gemma(image):
    """Ollama Gemma 3 4B model을 사용하여 이미지 분석"""
    try:
        # Convert Gradio image to PIL Image
        pil_image = Image.fromarray(image)   # PIL(이미지처리, 조작) 이미지 객체로 변환

        # Save image to a temporary buffer
        buffer = io.BytesIO()  # 메모리 상에 임시 버퍼를 만들어서 이미지 데이터 저장
        pil_image.save(buffer, format="PNG")

        # Encode image to base64
        encoded_image = base64.b64encode(buffer.getvalue()).decode('utf-8')  # 임시 버퍼에 저장된 이미지 데이터를 바이트 문자열로 가져옴

        # Perform image analysis using Ollama Gemma3
        response = ollama.chat(
            model='gemma3:4b',
            messages=[
                {
                    'role': 'user',
                    'content': '이미지의 내용을 자세히 설명해줘. 무엇이 보이는지, 색상, 구성, 감정 등을 포함해서 분석해줘.',
                    'images': [encoded_image]
                }
            ]
        )

        return response['message']['content']

    except Exception as e:
        return f"이미지 분석 중 오류 발생: {str(e)}"
    
iface = gr.Interface(
        fn=analyze_image_with_gemma,
        inputs=gr.Image(type="numpy", label="이미지 업로드"),
        outputs=gr.Textbox(label="이미지 분석 결과"),
        title="Ollama Gemma3 이미지 분석기",
        description="이미지를 업로드하면 Gemma 3 4B 모델이 분석해드립니다."
    )

iface.launch()

c:\NEWTEST\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## 챗봇 예제 (역할부여)

In [6]:
# 사전 설치 : pip install ollama
import ollama

# 챗봇의 기본적 질문, 답변 역할
def ask_gemma(question):
    # ollama를 사용하여 모델로부터 응답 생성
    chatbot_role = "너는 항상 민간요법에 기반된 비과학적인 대답을 해. 질문에 대한 답은 3줄 이내로 짧게해줘."
    response = ollama.chat(model='gemma2', messages=[
        {"role": "system", "content": chatbot_role},  # 챗봇의 기본 역할 부여
        {"role": "user", "content": question}, # 질문
    ])

    return response['message']['content']

question = "요즘 기분이 우울해."
response = ask_gemma(question)
print(response)

밤하늘을 보며 은성의 아름다움에 집중해봐요.  

따뜻한 차 한잔 마시고, 좋아하는 향수를 살짝 뿌려보세요.  아침에는 노란 꽃을 가득 담은 화분을 두어 에너지를 불어넣어 보세요.   





## 챗봇 예제 (Gradio)

In [7]:
# 사전 설치 : pip install gradio
from langchain_community.chat_models import ChatOllama
from langchain.schema import HumanMessage, AIMessage   # HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
import gradio as gr

# ChatOllama 모델 초기화
model = ChatOllama(model="gemma3:4b", temperature=0.7, verbose=False)
# temperture가 낮을수록 거의 동일답변, 높을수록 창의적인 답변

# 채팅 기록을 포함하여 응답을 생성하는 함수
def chat(message, history):
    # 이전 대화 기록을 ChatOllama 형식으로 변환
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 현재 메시지 추가
    chat_history.append(HumanMessage(content=message))

    # 모델을 사용하여 응답 생성
    response = model.invoke(chat_history)

    return response.content

# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "안녕하세요!",
        "인공지능에 대해 설명해주세요.",
        "파이썬의 장점은 무엇인가요?"
    ],
    title="AI 챗봇",
    description="질문을 입력하면 AI가 답변합니다."
)

# 서버 실행
demo.launch(server_port=7861, server_name="0.0.0.0")

C:\Users\humna-20\AppData\Local\Temp\ipykernel_3668\2839257629.py:7: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  model = ChatOllama(model="gemma3:4b", temperature=0.7, verbose=False)
c:\NEWTEST\myenv\lib\site-packages\gradio\chat_interface.py:334: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://0.0.0.0:7861

To create a public link, set `share=True` in `launch()`.


In [8]:
demo.close()

Closing server running on port: 7861


## 챗봇 예제(Gradio + csv)

In [9]:
import pandas as pd
from langchain_community.chat_models import ChatOllama
from langchain.schema import HumanMessage, AIMessage   # HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter  # 특정 문자(예: 줄바꿈, 공백)를 기준으로 텍스트를 분할하는 기능을 제공
from langchain.chains import ConversationalRetrievalChain  # 질문과 관련된 정보 검색, 이전 대화정보도 같이 LLM에 제공, 답변생성
import gradio as gr

# CSV 파일 로드
df = pd.read_csv("./dataset/indata_kor.csv", encoding='CP949')

# 텍스트 분할
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200) # 텍스트 조각 최대 1000자, 텍스트 조각 사이에 200자만큼의 중복을 허용(문맥 유지)
texts = text_splitter.split_text("\n".join(df.to_string()))   # 문자열들을 줄바꿈 문자(\n)를 기준으로 연결

# 임베딩 모델 초기화
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")
# 모델 이름 : 조직이름(sentence-transformers) 다양한 작업 가능(all)-MS사 경령화 트랜스포머모델(MiniLM)-모델의 레이어수(L6)-모델이 버전(v2)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 벡터 데이터베이스 생성
vectorstore = FAISS.from_texts(texts, embeddings)  # from_texts : 임베딩으로 변환된 벡터를 FAISS 인덱스에 저장

# ChatOllama 모델 초기화
llm = ChatOllama(model="gemma3:4b", tempeature=0.1)   # temperture가 낮을수록 거의 동일답변, 높을수록 창의적인 답변

qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    vectorstore.as_retriever(search_kwargs={"k":1}),  # 가장 관련성 높은 1개의 문서만 검색
    return_source_documents=True,   # 참고한 소스 문서 정보 반환 여부
    verbose=False   # 체인의 실행 과정 출력 여부
)

# 채팅 함수 정의
def chat(message, history):
    # 이전 대화 기록을 ConversationalRetrievalChain 형식으로 변환
    chat_history = [(human, ai) for human, ai in history]

    # 모델을 사용하여 응답 생성
    response = qa_chain({"question": message, "chat_history": chat_history})

    # 소스 문서 정보 추출
    sources = set([doc.metadata.get('source', 'Unknown') for doc in response['source_documents']])
    source_info = f"\n\n참고 출처: {', '.join(sources)}" if sources else ""

    return response['answer'] + source_info

# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "한국폴리텍대학 스마트금융과 면접시에는 어떤걸 준비하고 가면 될까요?",
        "스마트금융과에 대해 설명해주세요",
        "한국폴리텍대한 추천할만한 학과 하나를 소개해주세요."
    ],
    title="대학 정보 AI 챗봇",
    description="스마트금융과에 대한 질문을 입력하면 AI가 CSV데이터를 참고하여 한글로 답변합니다."
)

# 서버 실행
demo.launch(server_port=7861, server_name="0.0.0.0")

C:\Users\humna-20\AppData\Local\Temp\ipykernel_3668\676871360.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


c:\NEWTEST\myenv\lib\site-packages\gradio\chat_interface.py:334: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://0.0.0.0:7861

To create a public link, set `share=True` in `launch()`.


C:\Users\humna-20\AppData\Local\Temp\ipykernel_3668\676871360.py:41: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain({"question": message, "chat_history": chat_history})


In [10]:
demo.close()

Closing server running on port: 7861
